# Download GVZ (CBOE Gold Volatility Index) Daily Close

Pulls the full daily **close** history of **GVZ** (the CBOE Gold ETF Volatility Index, i.e. the "gold VIX") directly from CBOE's official history CSV and saves it to a Parquet file.

- **Source:** `https://cdn.cboe.com/api/global/us_indices/daily_prices/GVZ_History.csv` (official, free, no API key). This endpoint publishes the daily index *close* only.
- **History:** since 2009-09-18 through the latest trading day.
- **Output:** `GVZ_daily.parquet` (next to this notebook), with a `date` index and a single `close` column.

In [ ]:
# Cell 1 — Install dependencies
%pip install -q pandas requests pyarrow

In [ ]:
# Cell 2 — Connect to CBOE and download the GVZ daily close history
import pandas as pd
import requests
from io import StringIO

GVZ_URL = "https://cdn.cboe.com/api/global/us_indices/daily_prices/GVZ_History.csv"

# CBOE's CDN can reject default urllib/pandas user agents, so use a browser-like header.
HEADERS = {"User-Agent": "Mozilla/5.0 (research data download)"}

resp = requests.get(GVZ_URL, headers=HEADERS, timeout=30)
resp.raise_for_status()

# The endpoint returns two columns: DATE, GVZ (the daily close level).
raw = pd.read_csv(StringIO(resp.text))

df = raw.rename(columns={raw.columns[0]: "date", raw.columns[1]: "close"})
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date").sort_index()
df["close"] = df["close"].astype(float)

# Sanity output
print("Shape:", df.shape)
print("Date range:", df.index.min().date(), "->", df.index.max().date())
print("NaNs in close:", int(df["close"].isna().sum()))
display(df.head())
display(df.tail())

In [ ]:
# Cell 3 — Save to Parquet
OUT_PATH = "GVZ_daily.parquet"

df.to_parquet(OUT_PATH, engine="pyarrow")
print(f"Saved {len(df):,} rows to {OUT_PATH}")

# Round-trip check
check = pd.read_parquet(OUT_PATH)
print("Read-back shape:", check.shape)
assert check.shape == df.shape, "Round-trip shape mismatch!"
check.tail()

In [2]:
# Cell 5 — Download XAU/USD (spot gold) 5-minute close from Dukascopy and save to Parquet
%pip install -q dukascopy-python pyarrow

from datetime import datetime, timedelta
import pandas as pd
import dukascopy_python
from dukascopy_python.instruments import INSTRUMENT_FX_METALS_XAU_USD

START_DATE = datetime(2010, 6, 8)    # 10 years ago  (edit to change the window)
END_DATE   = datetime(2026, 6, 1)   # one week later

# Make END_DATE inclusive (fetch's `end` is the upper bound of the bar timestamps).
_end = END_DATE + timedelta(days=1)

raw = dukascopy_python.fetch(
    INSTRUMENT_FX_METALS_XAU_USD,
    dukascopy_python.INTERVAL_MIN_5,
    dukascopy_python.OFFER_SIDE_BID,
    START_DATE,
    _end,
)

# Close only; Dukascopy timestamps are UTC.
xau = raw[["close"]].copy()
xau.index.name = "timestamp"

OUT_PATH = "XAU_USD_5_min_tick.parquet"
xau.to_parquet(OUT_PATH, engine="pyarrow")

check = pd.read_parquet(OUT_PATH)
print(f"Saved {len(xau):,} rows to {OUT_PATH}")
print("Range (UTC):", xau.index.min(), "->", xau.index.max())
print("Round-trip shape:", check.shape, "==", xau.shape, ":", check.shape == xau.shape)
check.tail()


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


INFO:DUKASCRIPT:current timestamp :2010-07-06T20:35:00
INFO:DUKASCRIPT:current timestamp :2010-08-04T15:55:00
INFO:DUKASCRIPT:current timestamp :2010-09-02T14:00:00
INFO:DUKASCRIPT:current timestamp :2010-10-01T10:05:00
INFO:DUKASCRIPT:current timestamp :2010-11-01T01:10:00
INFO:DUKASCRIPT:current timestamp :2010-11-30T05:20:00
INFO:DUKASCRIPT:current timestamp :2010-12-29T12:30:00
INFO:DUKASCRIPT:current timestamp :2011-01-27T12:50:00
INFO:DUKASCRIPT:current timestamp :2011-02-25T11:55:00
INFO:DUKASCRIPT:current timestamp :2011-03-28T06:00:00
INFO:DUKASCRIPT:current timestamp :2011-04-25T23:50:00
INFO:DUKASCRIPT:current timestamp :2011-05-23T12:55:00
INFO:DUKASCRIPT:current timestamp :2011-06-20T16:40:00
INFO:DUKASCRIPT:current timestamp :2011-07-18T21:05:00
INFO:DUKASCRIPT:current timestamp :2011-08-16T02:30:00
INFO:DUKASCRIPT:current timestamp :2011-09-13T10:15:00
INFO:DUKASCRIPT:current timestamp :2011-10-11T17:10:00
INFO:DUKASCRIPT:current timestamp :2011-11-09T17:20:00
INFO:DUKAS

Saved 1,149,284 rows to XAU_USD_5_min_tick.parquet
Range (UTC): 2010-06-07 23:00:00+00:00 -> 2026-06-01 23:00:00+00:00
Round-trip shape: (1149284, 1) == (1149284, 1) : True


,close
timestamp,
2026-06-01 22:40:00+00:00,4481.395
2026-06-01 22:45:00+00:00,4483.925
2026-06-01 22:50:00+00:00,4483.695
2026-06-01 22:55:00+00:00,4485.388
2026-06-01 23:00:00+00:00,4485.748


In [15]:
df_check = pd.read_parquet(OUT_PATH)

df_check.head(20)



,close
timestamp,
2010-06-07 23:00:00+00:00,1239.472
2010-06-07 23:05:00+00:00,1239.430
2010-06-07 23:10:00+00:00,1239.374
2010-06-07 23:15:00+00:00,1239.471
2010-06-07 23:20:00+00:00,1239.914
2010-06-07 23:25:00+00:00,1239.173
2010-06-07 23:30:00+00:00,1239.252
2010-06-07 23:35:00+00:00,1239.494
2010-06-07 23:40:00+00:00,1239.486


close    1239.43
Name: 2010-06-07 23:05:00+00:00, dtype: float64